# EDA — CFI-G (PSI-159): ignition and self-extinction of spherical gas flames

Same method as `01_eda_flex.ipynb`: source first, then analysis → conclusions → decision → documentation. Decisions are logged in the shared Drive (`NOTAS.md`, `docs/DECISIONES_DATOS.md`). Since 2026-11-14 Claude decides non-controversial points (autonomous mode agreed with LC) and marks them as such.

## 1. Load and provenance
- `PSI-159_Experimental table_CFI-G_Normal Flames.csv` (fuel flows out of the burner into an O₂/N₂ atmosphere) and `..._Inverse Flames.csv` (O₂/N₂ flows out of the burner into a fuel/N₂ atmosphere). Downloaded from the PSI API (2026-11-13).
- NASA report: *ACME/CFI-G Final Report* (21 Nov 2022), `datos/documentos_ciencia/PSI-159/FinalRpt_ACME_CFI-G.pdf` (93 pp.).

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, re
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
DATA = r'G:/Mi unidad/Hackatones/Nasa space apps 2026/datos/tablas_experimentales/'
rn = pd.read_csv(DATA + 'PSI-159_Experimental table_CFI-G_Normal Flames.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
ri = pd.read_csv(DATA + 'PSI-159_Experimental table_CFI-G_Inverse Flames.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
print('normal:', rn.shape, '| inverse:', ri.shape, '-> fully empty trailing rows in inverse:', int((ri['Test'].str.strip() == '').sum()))
ri = ri[ri['Test'].str.strip() != ''].copy()
print('inverse after dropping fully empty rows:', ri.shape)

normal: (179, 18) | inverse: (1015, 20) -> fully empty trailing rows in inverse: 861
inverse after dropping fully empty rows: (154, 20)


## 2. What the NASA report says (read before interpreting the table)

- **Outcomes.** 280 hot flames were observed; **189 went out on their own by radiative heat loss** (`SE`, self-extinction) and the rest were **stopped by the operator** (`FT`, flow termination) "because the preset flow time was out or the burner overheated" (§4.K). So `FT` means *the flame was still alive when the test ended*: a **censored** outcome (we do not know whether it would have gone out later), not "it does not go out".
- **Cool flames** (the scientific goal) appeared only in 15 cases: normal flames, 30 % n-butane, 40 % O₂, 2.03 bar, 0.75–1.5 mg/s. Too few to model; they are evidence.
- **Not every "no ignition" is about the gas mixture.** Some tests deliberately used a **low ignition energy** to try to ignite a cool flame directly (GMT 102 and 106; igniter voltage in the comments; "direct cool – no hot ignition"). On **30 Aug 2021 (day 21242)** the igniter tip and the image unit failed: "three tests failed to ignite" because of hardware.
- **`TFP` = Thin Filament Pyrometry**: whether the fiber array used to measure temperature was in place. A diagnostic, not a condition or an outcome (resolves the open question in `datos/CSV.md`).
- **The atmosphere is nominal**: during a day, pressure rises and O₂ drops as combustion products accumulate ("vitiation"); hence `XO2 before/after test`.
- `Tad` (adiabatic flame temperature) and `Zst` (stoichiometric mixture fraction) are **computed** from the conditions, not measured.
- Mass-flow controllers are inaccurate below ~10 % of their range; at high flows the flame is not spherical.

In [2]:
def num(s):
    s = s.astype(str).str.strip().replace({'-': '', '#VALUE!': ''})
    return pd.to_numeric(s, errors='coerce')

def base_fuel(s):
    return s.str.lower().str.extract(r'(ethane|propane|butane)')[0]

n = pd.DataFrame({
    'flame_type': 'normal', 'day_no': num(rn['Day']), 'test_id': rn['Test'].str.replace(' ', '', regex=False),
    'fuel': base_fuel(rn['Fuel']),
    'fuel_bottle_frac': num(rn['Fuel'].str.extract(r'(\d+)%')[0]).div(100).fillna(1.0),
    'o2_frac': num(rn[' XO2 before test']), 'o2_frac_after': num(rn['XO2  after test']),
    'pressure_bar': num(rn['approx p (bar)']),
    'jet_fuel_frac': num(rn['Xf']), 'fuel_flow_mg_s': num(rn['mHC (mg/s)']), 'o2_flow_mg_s': np.nan,
    'atm_fuel_frac': np.nan,
    'tfp': rn['TFP'].str.strip(), 'tad_k': num(rn['Tad (K)']), 'zst': num(rn['Zst']),
    'hot_ignition_raw': rn['Hot ignition'].str.strip(), 'burn_time_s': num(rn['burn time (sec)']),
    'end_raw': rn['Self-ext or flow term'].str.strip(), 'peak_burner_c': num(rn['Peak burner T (C )']),
    'cool_flame_raw': rn['Cool flame?'].str.strip(), 'color': '', 'comments': rn['Comments'].str.strip()})
i = pd.DataFrame({
    'flame_type': 'inverse', 'day_no': num(ri['Day']), 'test_id': ri['Test'].str.replace(' ', '', regex=False),
    'fuel': base_fuel(ri['Fuel']), 'fuel_bottle_frac': np.nan,
    'o2_frac': num(ri['XO2']), 'o2_frac_after': np.nan, 'pressure_bar': num(ri['approx p (bar)']),
    'jet_fuel_frac': np.nan, 'fuel_flow_mg_s': num(ri['HC burnrate (mg/s)']), 'o2_flow_mg_s': num(ri['O2 flow rate (mg/s)']),
    'atm_fuel_frac': num(ri['Xf  before test']),
    'tfp': ri['TFP'].str.strip(), 'tad_k': num(ri['Tad (K)']), 'zst': num(ri['Zst']),
    'hot_ignition_raw': ri['Hot ignition'].str.strip(), 'burn_time_s': num(ri['burn time (sec)']),
    'end_raw': ri['Self -ext or flow term'].str.strip(), 'peak_burner_c': num(ri['Peak burner Temp (C)']),
    'cool_flame_raw': '', 'color': ri['Color'].str.strip(), 'comments': ri['Comments'].str.strip()})
df = pd.concat([n, i], ignore_index=True)
df['test_day'] = df['test_id'].str[:5]
df['ignition'] = df['hot_ignition_raw'].str.lower().map({'yes': 'yes', 'no': 'no', 'flash': 'flash'})
df['end'] = df['end_raw'].map({'SE': 'self_ext', 'FT': 'flow_term', '-': 'none'})
print(df.shape)
print(df.groupby('flame_type')[['ignition', 'end']].value_counts(dropna=False).to_string())

(333, 25)
flame_type  ignition  end      
inverse     yes       self_ext      73
                      flow_term     58
            no        none          15
            flash     self_ext       8
normal      yes       self_ext     102
            no        none          46
            yes       flow_term     30
            no        flow_term      1


In [3]:
# Consistency between the two outcome columns and the burn time
print(pd.crosstab([df['flame_type'], df['ignition']], df['end'].fillna('NA')))
print('\nburn time by outcome:'); print(df.groupby(['flame_type', 'end'])['burn_time_s'].agg(['median', 'count', 'max']))
print('\nduplicated test ids:', df['test_id'][df['test_id'].duplicated()].tolist())
print('fuel not parsed:', int(df['fuel'].isna().sum()), '| test days:', df['test_day'].nunique())

end                  flow_term  none  self_ext
flame_type ignition                           
inverse    flash             0     0         8
           no                0    15         0
           yes              58     0        73
normal     no                1    46         0
           yes              30     0       102

burn time by outcome:
                      median  count    max
flame_type end                            
inverse    flow_term    66.0     58  152.0
           none          0.0     15    0.0
           self_ext      8.0     81   55.0
normal     flow_term    61.0     31  276.0
           none          0.0     42    0.0
           self_ext     29.5    102  304.0

duplicated test ids: []
fuel not parsed: 0 | test days: 23


In [4]:
# Tests where "no ignition" is not about the gas mixture (report §3.B, comments)
df['low_energy_ignition'] = df['comments'].str.contains('Igniter voltage|direct cool', case=False) & (df['ignition'] != 'yes')
df['hardware_issue'] = df['comments'].str.contains('abort|data lost|No nitrogen', case=False) | ((df['test_day'] == '21242') & (df['ignition'] != 'yes'))
print('deliberate low-energy ignition attempts:', int(df['low_energy_ignition'].sum()), df.loc[df['low_energy_ignition'], 'test_id'].tolist())
print('hardware problems:'); print(df.loc[df['hardware_issue'], ['test_id', 'ignition', 'end', 'comments']].to_string(index=False))
print('\nnormal flames: ignition by jet fuel fraction (Xf):')
nn = df[df.flame_type == 'normal']
print(pd.crosstab(nn['jet_fuel_frac'].round(3), nn['ignition']))

deliberate low-energy ignition attempts: 9 ['21102C4', '21102C6', '21102C8', '21102M3', '21102M4', '21102M5', '21102M6', '21106F2', '21106F5']
hardware problems:
test_id ignition      end                               comments
21209A2       no     none                                Aborted
21242F2       no     none F1 was aborted, and F2 did not ignite.
21242F3       no     none                                       
21242F4       no     none                                       
21242P2       no     none               Most data lost this day.
21256C2      yes self_ext   No nitrogen flowed. This repeats A3.

normal flames: ignition by jet fuel fraction (Xf):
ignition       no  yes
jet_fuel_frac         
0.000           2    0
0.018           1    0
0.024           1    0
0.027          10    0
0.031           0    4
0.045           4    2
0.047           8    8
0.080           0    1
0.125           0    2
0.150           0    7
0.250           0    4
0.300           2   65
0.500    

### 2.1 Findings on sources and outcome labels
- 333 tests (179 normal + 154 inverse); 861 fully empty rows at the end of the inverse file (spreadsheet residue, not tests). No duplicated ids; 23 test days.
- The two outcome columns agree with each other and with the burn time, with **one exception**: `21106F4` says *no hot ignition* but has `FT`, 22 s of burn and the comment "hot flame". Strong evidence that it did ignite.
- `flash` (8 inverse tests) = a momentary flame that went out at once (`SE`, ~1 s). For ignition it counts as **ignited**; for self-extinction it is an immediate self-extinction.
- 9 tests are deliberate **low-energy ignition** attempts (cool-flame trials) and 5 failed or aborted because of **hardware** (21242 igniter/IPSU failure, 21209A2 aborted). `21256C2` ran with *no nitrogen flowing*, so its recorded conditions are wrong.
- In normal flames, **ignition depends strongly on how diluted the fuel jet is**: jets with ≤ 2.7 % fuel never ignited, 3–5 % is mixed, ≥ 8 % almost always ignites.

### 2.2 Decisions (Claude, autonomous mode; logged as DD-026 to DD-028)
- **DD-026:** `21106F4` → ignition = yes, flag `corrected_label`, original kept in `hot_ignition_raw`.
- **DD-027:** low-energy attempts and hardware failures are **kept as rows but excluded from the ignition model** (flags `low_energy_ignition`, `hardware_issue`): their "no ignition" says nothing about the gas mixture. `21256C2` (wrong conditions) is excluded from every model.
- **DD-028:** `flash` counts as ignited, followed by immediate self-extinction. `TFP`, `Tad`, `Zst` and `Color` are not inputs (diagnostic / computed / observed after the test).

In [5]:
df.loc[df['test_id'] == '21106F4', 'ignition'] = 'yes'
df['corrected_label'] = df['test_id'] == '21106F4'
df['conditions_wrong'] = df['test_id'] == '21256C2'
df['ignited'] = df['ignition'].isin(['yes', 'flash'])
df['self_ext'] = df['end'] == 'self_ext'
df['ign_model'] = ~df['low_energy_ignition'] & ~df['hardware_issue'] & ~df['conditions_wrong']
df['ext_model'] = df['ignited'] & ~df['conditions_wrong'] & df['end'].isin(['self_ext', 'flow_term'])
print('ignition model:', int(df['ign_model'].sum()), 'tests ->', df.loc[df['ign_model'], 'ignited'].value_counts().to_dict())
print('self-extinction data (ignited):', int(df['ext_model'].sum()), 'tests ->', df.loc[df['ext_model'], 'end'].value_counts().to_dict())
print(df.loc[df['ext_model']].groupby('flame_type')['end'].value_counts().unstack())

ignition model: 318 tests -> {True: 271, False: 47}
self-extinction data (ignited): 271 tests -> {'self_ext': 182, 'flow_term': 89}
end         flow_term  self_ext
flame_type                     
inverse            58        80
normal             31       102


## 3. Conditions: ranges and coverage

In [6]:
print(df.groupby('flame_type')[['o2_frac', 'pressure_bar', 'jet_fuel_frac', 'atm_fuel_frac', 'fuel_flow_mg_s', 'o2_flow_mg_s', 'burn_time_s', 'peak_burner_c']].describe().T.round(3).to_string())
print('\nnormal: O2 drop during a day (before - after):', (df['o2_frac'] - df['o2_frac_after']).describe().round(4)[['mean', 'max']].to_dict())
for ft in ['normal', 'inverse']:
    d = df[df.flame_type == ft]
    print(f'\n{ft}: tests by fuel x pressure'); print(pd.crosstab(d['fuel'], d['pressure_bar'].round(2)))
    print(f'{ft}: tests by fuel x O2 (rounded)'); print(pd.crosstab(d['fuel'], (d['o2_frac'] * 100).round(-1).astype(int)))

flame_type            inverse   normal
o2_frac        count  154.000  179.000
               mean     0.589    0.322
               std      0.287    0.083
               min      0.120    0.180
               25%      0.232    0.212
               50%      0.850    0.376
               75%      0.850    0.390
               max      0.850    0.400
pressure_bar   count  154.000  179.000
               mean     1.732    1.279
               std      1.057    0.675
               min      0.495    0.507
               25%      0.552    0.944
               50%      1.103    1.013
               75%      2.207    2.027
               max      3.310    3.040
jet_fuel_frac  count    0.000  179.000
               mean       NaN    0.371
               std        NaN    0.306
               min        NaN    0.000
               25%        NaN    0.150
               50%        NaN    0.300
               75%        NaN    0.500
               max        NaN    1.000
atm_fuel_frac  count  154


normal: O2 drop during a day (before - after): {'mean': 0.0024, 'max': 0.0231}

normal: tests by fuel x pressure
pressure_bar  0.51  0.94  1.01  2.03  3.04
fuel                                      
butane          19     0    46    17     0
ethane           6    10    17     8     0
propane          0    12    20    12    12
normal: tests by fuel x O2 (rounded)
o2_frac  20  30  40
fuel               
butane   17   0  65
ethane   12   3  26
propane  35   0  21



inverse: tests by fuel x pressure
pressure_bar  0.50  0.55  1.10  2.21  3.31
fuel                                      
butane           0    14    10    17    17
ethane           6    12    17     9     9
propane          0     8    15    10    10
inverse: tests by fuel x O2 (rounded)
o2_frac  10  20  30  40  50  70  80
fuel                               
butane    0   8   3   4  10   1  32
ethane    2  17   4   0   3   0  27
propane   0  12   2   0   9   0  20


## 4. Outcomes vs conditions

In [7]:
nm = df[(df.flame_type == 'normal')]
print('NORMAL — ignition share (ignition-model tests) by O2 band and fuel-jet band:')
g = nm[nm['ign_model']]
print(pd.crosstab(pd.cut(g['o2_frac'], [0.15, 0.25, 0.32, 0.45]), pd.cut(g['jet_fuel_frac'], [-0.01, 0.03, 0.06, 0.2, 1.0]), values=g['ignited'], aggfunc='mean').round(2))
print(pd.crosstab(pd.cut(g['o2_frac'], [0.15, 0.25, 0.32, 0.45]), pd.cut(g['jet_fuel_frac'], [-0.01, 0.03, 0.06, 0.2, 1.0])))
print('\nNORMAL — self-extinction share among ignited, by O2 band and pressure:')
e = nm[nm['ext_model']]
print(pd.crosstab(pd.cut(e['o2_frac'], [0.15, 0.25, 0.32, 0.45]), e['pressure_bar'].round(2), values=e['self_ext'], aggfunc='mean').round(2))
print(pd.crosstab(pd.cut(e['o2_frac'], [0.15, 0.25, 0.32, 0.45]), e['pressure_bar'].round(2)))
print('\nNORMAL — self-extinction share by fuel and by fuel-flow band:')
print(e.groupby('fuel')['self_ext'].agg(['size', 'mean']).round(2))
print(e.groupby(pd.cut(e['fuel_flow_mg_s'], [0, 0.6, 1.1, 2, 4]), observed=True)['self_ext'].agg(['size', 'mean']).round(2))
print('\nburn time of self-extinguished vs flow-terminated (normal):'); print(e.groupby('end')['burn_time_s'].describe()[['count', '25%', '50%', '75%']])

NORMAL — ignition share (ignition-model tests) by O2 band and fuel-jet band:
jet_fuel_frac  (-0.01, 0.03]  (0.03, 0.06]  (0.06, 0.2]  (0.2, 1.0]
o2_frac                                                            
(0.15, 0.25]             NaN          0.45          1.0        0.81
(0.32, 0.45]             0.0          1.00          1.0        0.95
jet_fuel_frac  (-0.01, 0.03]  (0.03, 0.06]  (0.06, 0.2]  (0.2, 1.0]
o2_frac                                                            
(0.15, 0.25]               0            22            1          37
(0.32, 0.45]              14             4            9          83

NORMAL — self-extinction share among ignited, by O2 band and pressure:
pressure_bar  0.51  0.94  1.01  2.03  3.04
o2_frac                                   
(0.15, 0.25]  1.00   NaN  1.00   1.0   1.0
(0.32, 0.45]  0.72   0.5  0.77   0.6   NaN
pressure_bar  0.51  0.94  1.01  2.03  3.04
o2_frac                                   
(0.15, 0.25]     3     0    23     7     8
(0.32,

In [8]:
iv = df[(df.flame_type == 'inverse')]
print('INVERSE — ignition (ignition-model tests) by jet O2 and fuel in the atmosphere:')
g = iv[iv['ign_model']]
print(pd.crosstab(g['o2_frac'], pd.cut(g['atm_fuel_frac'], [0, 0.08, 0.15, 0.3, 0.5]), values=g['ignited'], aggfunc='mean').round(2))
print('\nINVERSE — self-extinction share among ignited, by jet O2 and pressure:')
e = iv[iv['ext_model']]
print(pd.crosstab(e['o2_frac'], e['pressure_bar'].round(2), values=e['self_ext'], aggfunc='mean').round(2))
print(pd.crosstab(e['o2_frac'], e['pressure_bar'].round(2)))

INVERSE — ignition (ignition-model tests) by jet O2 and fuel in the atmosphere:
atm_fuel_frac  (0.0, 0.08]  (0.08, 0.15]  (0.15, 0.3]  (0.3, 0.5]
o2_frac                                                          
0.120                  NaN           NaN         0.00         NaN
0.135                  NaN           NaN         1.00         NaN
0.150                  NaN           NaN         1.00         NaN
0.180                  NaN           NaN         1.00         NaN
0.190                  NaN           NaN         1.00         NaN
0.210                 0.57           1.0         0.78        0.00
0.300                  NaN           1.0         1.00        0.67
0.400                 1.00           NaN          NaN         NaN
0.450                 1.00           NaN          NaN         NaN
0.500                 1.00           1.0         1.00        1.00
0.700                 1.00           NaN          NaN         NaN
0.850                 0.95           1.0         1.00        1

pressure_bar  0.50  0.55  1.10  2.21  3.31
o2_frac                                   
0.135            0     0     1     0     0
0.150            1     0     6     0     0
0.180            0     1     1     0     0
0.190            0     0     1     0     0
0.210            2     3     5     3     4
0.300            3     4     0     0     0
0.400            0     0     0     0     3
0.450            0     0     0     0     1
0.500            0     4     5     6     6
0.700            0     0     0     0     1
0.850            0    14    20    25    18


In [9]:
# Is "flow terminated" just "the operator stopped early"? How long had FT flames burned?
e = df[df['ext_model']]
for T in [10, 20, 30]:
    usable = e[(e['end'] == 'self_ext') | (e['burn_time_s'] >= T)]
    lab = (usable['end'] == 'self_ext') & (usable['burn_time_s'] < T)
    print(f'"goes out within {T} s": usable {len(usable)} of {len(e)} | went out {int(lab.sum())} | lost (FT shorter than {T} s): {int(((e["end"] == "flow_term") & (e["burn_time_s"] < T)).sum())}')
print('\nFT burn time (s) by flame type:'); print(e[e['end'] == 'flow_term'].groupby('flame_type')['burn_time_s'].describe()[['count', 'min', '25%', '50%', 'max']])
print('\nFT comments:'); print(e.loc[e['end'] == 'flow_term', 'comments'].value_counts().head(10))

"goes out within 10 s": usable 271 of 271 | went out 76 | lost (FT shorter than 10 s): 0
"goes out within 20 s": usable 265 of 271 | went out 111 | lost (FT shorter than 20 s): 6
"goes out within 30 s": usable 254 of 271 | went out 124 | lost (FT shorter than 30 s): 17

FT burn time (s) by flame type:
            count   min   25%   50%    max
flame_type                                
inverse      58.0  12.0  43.0  66.0  152.0
normal       31.0  13.0  31.5  61.0  276.0

FT comments:
comments
                                                                  59
hit the temperature limit                                          6
hot flame                                                          2
Irregular Oscillation                                              2
Hit the burner temperature limit                                   2
FC=flow cut                                                        1
Lowest flow rate for ignition                                      1
Igniter voltage: 2.

### 4.1 Conclusions (for discussion)

1. **Ignition is mostly about how diluted the fuel is.** Normal flames: a jet with ≤ 3 % fuel never ignited; with ≥ 6 % it almost always did. Inverse flames: almost everything ignited (the few failures are at 21 % O₂ with very little or a lot of fuel around). Only 47 real "no ignition" cases (all after removing deliberate low-energy and hardware cases), most of them on two days. Useful, but it says little about fire safety beyond "too little fuel does not light".
2. **Self-extinction is the interesting result for fire safety.** In normal flames at ~21 % O₂, **every** flame that ignited went out by itself (41 of 41); at ~40 % O₂ only ~70 % did, the rest were still burning when the operator stopped them. In inverse flames, pressure matters most: at ~0.5 bar almost none went out, at 2–3 bar almost all did (bigger flames lose more heat by radiation).
3. **The catch: `FT` is censored.** A flow-terminated flame was stopped by a timer or by burner overheating, not because it "never goes out". FT flames had burned longer (median 61–66 s) than self-extinguished ones (median 8–30 s). Treating FT as "did not go out" would bias the model towards "flames persist" wherever the operator used short timers.
4. Normal and inverse flames cannot share one model: in one O₂ is the atmosphere, in the other it is the burner jet. They are two experiments.
5. Coverage: normal flames have O₂ only around 21 % and 40 % (almost nothing at 30 %), pressure 0.5–3 bar; butane never at 3 bar, propane never at 0.5 bar.

**Decision needed from LC (how to model CFI-G, given the time left):**
- (a) **"Does the flame go out by itself within T seconds?"** (logistic, like FLEX): FT flames that burned less than T are dropped (we do not know their ending). Simple and honest; loses a few tests.
- (b) **Time until self-extinction** (survival model, FT = censored): the most correct, but a new kind of model to build and explain.
- (c) **CFI-G only as evidence** (no model): spend the time on BASS/BASS-II (cabin materials), which are closer to the challenge.
- The ignition model (point 1) is cheap but of little value; Claude suggests leaving it as evidence.